# Lan truyền xuôi, Lan truyền ngược và Đồ thị tính toán
:label:`sec_backprop`

Cho đến nay, chúng ta đã huấn luyện các mô hình của mình bằng phương pháp hạ cực âm gradient ngẫu nhiên theo mini-batch (minibatch stochastic gradient descent).
Tuy nhiên, khi triển khai thuật toán, chúng ta chỉ quan tâm đến các phép tính liên quan đến *lan truyền xuôi* (forward propagation) qua mô hình.
Khi đến lúc tính toán các gradient, chúng ta chỉ cần gọi hàm lan truyền ngược được cung cấp bởi khung phát triển học sâu (deep learning framework).

Việc tự động tính toán các gradient làm đơn giản hóa sâu sắc việc triển khai các thuật toán học sâu.
Trước khi có vi phân tự động, ngay cả những thay đổi nhỏ đối với các mô hình phức tạp cũng yêu cầu tính toán lại các đạo hàm phức tạp bằng tay.
Rất thường xuyên, các bài báo học thuật đã phải dành nhiều trang để rút ra các quy tắc cập nhật.
Mặc dù chúng ta phải tiếp tục dựa vào vi phân tự động để có thể tập trung vào những phần thú vị,
nhưng bạn cần biết cách các gradient này được tính toán dưới mui xe nếu bạn muốn đi sâu hơn một sự hiểu biết nông cạn về học sâu.

Trong phần này, chúng ta sẽ đi sâu vào chi tiết của *lan truyền ngược* (backward propagation - thường được gọi là *backpropagation*).
Để truyền đạt một số hiểu biết sâu sắc cho cả kỹ thuật và triển khai của chúng, chúng tôi dựa trên một số toán học cơ bản và đồ thị tính toán.
Để bắt đầu, chúng tôi tập trung trình bày vào một MLP một lớp ẩn với suy giảm trọng số (weight decay - chuẩn hóa $\ell_2$, sẽ được mô tả trong các chương tiếp theo).

## Lan truyền xuôi

*Lan truyền xuôi* (hoặc *lượt truyền xuôi*) đề cập đến việc tính toán và lưu trữ các biến trung gian (bao gồm cả đầu ra) cho một mạng thần kinh theo thứ tự từ lớp đầu vào đến lớp đầu ra.
Bây giờ chúng ta sẽ làm việc từng bước qua cơ chế của một mạng thần kinh có một lớp ẩn.
Điều này có vẻ tẻ nhạt nhưng nói theo lời của nghệ sĩ funk virtuoso James Brown, bạn phải "trả giá để làm chủ" (pay the cost to be the boss).

Để đơn giản, chúng ta giả định rằng ví dụ đầu vào là $\mathbf{x}\in \mathbb{R}^d$ và lớp ẩn của chúng ta không bao gồm số hạng định thiên (bias).
Ở đây, biến trung gian là:

$$\mathbf{z}= \mathbf{W}^{(1)} \mathbf{x},$$

trong đó $\mathbf{W}^{(1)} \in \mathbb{R}^{h \times d}$ là tham số trọng số của lớp ẩn.
Sau khi chạy biến trung gian $\mathbf{z}\in \mathbb{R}^h$ qua hàm kích hoạt $\phi$, chúng ta thu được vectơ kích hoạt ẩn có độ dài $h$:

$$\mathbf{h}= \phi (\mathbf{z}).$$

Đầu ra của lớp ẩn $\mathbf{h}$ cũng là một biến trung gian.
Giả sử rằng các tham số của lớp đầu ra chỉ sở hữu một trọng số là $\mathbf{W}^{(2)} \in \mathbb{R}^{q \times h}$, chúng ta có thể thu được một biến lớp đầu ra với một vectơ có độ dài $q$:

$$\mathbf{o}= \mathbf{W}^{(2)} \mathbf{h}.$$

Giả sử rằng hàm mất mát là $l$ và nhãn ví dụ là $y$, chúng ta có thể tính toán số hạng mất mát cho một ví dụ dữ liệu đơn lẻ:

$$L = l(\mathbf{o}, y).$$

Như chúng ta sẽ thấy định nghĩa về chuẩn hóa $\ell_2$ sẽ được giới thiệu sau, với siêu tham số $\lambda$, số hạng chuẩn hóa là:

$$s = \frac{\lambda}{2} \left(\|\mathbf{W}^{(1)}\|_\textrm{F}^2 + \|\mathbf{W}^{(2)}\|_\textrm{F}^2\right),$$
:eqlabel:`eq_forward-s`

trong đó chuẩn Frobenius của ma trận đơn giản là chuẩn $\ell_2$ được áp dụng sau khi làm phẳng ma trận thành một vectơ.
Cuối cùng, mất mát có chuẩn hóa của mô hình trên một ví dụ dữ liệu nhất định là:

$$J = L + s.$$

Chúng ta gọi $J$ là *hàm mục tiêu* (objective function) trong cuộc thảo luận tiếp theo.

## Đồ thị tính toán của lan truyền xuôi

Vẽ *đồ thị tính toán* giúp chúng ta hình dung các phụ thuộc của các toán tử và biến trong phép tính.
Hình :numref:`fig_forward` chứa đồ thị liên kết với mạng đơn giản được mô tả ở trên, trong đó các hình vuông biểu thị các biến và các hình tròn biểu thị các toán tử.
Góc dưới bên trái biểu thị đầu vào và góc trên bên phải là đầu ra.
Lưu ý rằng các hướng của mũi tên (minh họa luồng dữ liệu) chủ yếu là hướng sang phải và hướng lên trên.

![Đồ thị tính toán của lan truyền xuôi.](http://d2l.ai/_images/forward.svg)
:label:`fig_forward`

## Lan truyền ngược

*Lan truyền ngược* đề cập đến phương pháp tính toán gradient của các tham số mạng thần kinh.
Tóm lại, phương pháp này duyệt qua mạng theo thứ tự ngược lại, từ lớp đầu ra đến lớp đầu vào, theo *quy tắc chuỗi* (chain rule) từ giải tích.
Thuật toán lưu trữ bất kỳ biến trung gian nào (đạo hàm riêng) cần thiết trong khi tính toán gradient đối với một số tham số.
Giả sử rằng chúng ta có các hàm $\mathsf{Y}=f(\mathsf{X})$ và $\mathsf{Z}=g(\mathsf{Y})$, trong đó đầu vào và đầu ra $\mathsf{X}, \mathsf{Y}, \mathsf{Z}$ là các tensor có hình dạng tùy ý.
Bằng cách sử dụng quy tắc chuỗi, chúng ta có thể tính đạo hàm của $\mathsf{Z}$ đối với $\mathsf{X}$ thông qua:

$$\frac{\partial \mathsf{Z}}{\partial \mathsf{X}} = \textrm{prod}\left(\frac{\partial \mathsf{Z}}{\partial \mathsf{Y}}, \frac{\partial \mathsf{Y}}{\partial \mathsf{X}}\right).$$

Ở đây chúng ta sử dụng toán tử $\textrm{prod}$ để nhân các đối số của nó sau khi thực hiện các phép toán cần thiết, chẳng hạn như chuyển vị và hoán đổi vị trí đầu vào.
Đối với các vectơ, điều này rất đơn giản: nó chỉ là phép nhân ma trận - ma trận.
Đối với các tensor có số chiều cao hơn, chúng ta sử dụng đối chiếu thích hợp.
Toán tử $\textrm{prod}$ ẩn đi tất cả các ký hiệu rườm rà.

Hãy nhớ lại rằng các tham số của mạng đơn giản có một lớp ẩn, có đồ thị tính toán trong Hình :numref:`fig_forward`, là $\mathbf{W}^{(1)}$ và $\mathbf{W}^{(2)}$.
Mục tiêu của lan truyền ngược là tính toán các gradient $\partial J/\partial \mathbf{W}^{(1)}$ và $\partial J/\partial \mathbf{W}^{(2)}$.
Để thực hiện điều này, chúng ta áp dụng quy tắc chuỗi và lần lượt tính toán gradient của từng biến trung gian và tham số.
Thứ tự tính toán được đảo ngược so với các phép tính được thực hiện trong lan truyền xuôi, vì chúng ta cần bắt đầu với kết quả của đồ thị tính toán và hoạt động theo cách của chúng ta đối với các tham số.
Bước đầu tiên là tính toán các gradient của hàm mục tiêu $J=L+s$ đối với số hạng mất mát $L$ và số hạng chuẩn hóa $s$:

$$\frac{\partial J}{\partial L} = 1 \; \textrm{và} \; \frac{\partial J}{\partial s} = 1.$$

Tiếp theo, chúng ta tính toán gradient của hàm mục tiêu đối với biến của lớp đầu ra $\mathbf{o}$ theo quy tắc chuỗi:

$$
\frac{\partial J}{\partial \mathbf{o}}
= \textrm{prod}\left(\frac{\partial J}{\partial L}, \frac{\partial L}{\partial \mathbf{o}}\right)
= \frac{\partial L}{\partial \mathbf{o}}
\in \mathbb{R}^q.
$$

Tiếp theo, chúng ta tính toán các gradient của số hạng chuẩn hóa đối với cả hai tham số:

$$\frac{\partial s}{\partial \mathbf{W}^{(1)}} = \lambda \mathbf{W}^{(1)}
\; \textrm{và} \;
\frac{\partial s}{\partial \mathbf{W}^{(2)}} = \lambda \mathbf{W}^{(2)}.$$

Bây giờ chúng ta có thể tính toán gradient $\partial J/\partial \mathbf{W}^{(2)} \in \mathbb{R}^{q \times h}$ của các tham số mô hình gần lớp đầu ra nhất.
Sử dụng quy tắc chuỗi ta được:

$$\frac{\partial J}{\partial \mathbf{W}^{(2)}}= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{o}}, \frac{\partial \mathbf{o}}{\partial \mathbf{W}^{(2)}}\right) + \textrm{prod}\left(\frac{\partial J}{\partial s}, \frac{\partial s}{\partial \mathbf{W}^{(2)}}\right)= \frac{\partial J}{\partial \mathbf{o}} \mathbf{h}^\top + \lambda \mathbf{W}^{(2)}.$$
:eqlabel:`eq_backprop-J-h`

Để có được gradient đối với $\mathbf{W}^{(1)}$, chúng ta cần tiếp tục lan truyền ngược dọc theo lớp đầu ra đến lớp ẩn.
Gradient đối với đầu ra lớp ẩn $\partial J/\partial \mathbf{h} \in \mathbb{R}^h$ được cho bởi:

$$
\frac{\partial J}{\partial \mathbf{h}}
= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{o}}, \frac{\partial \mathbf{o}}{\partial \mathbf{h}}\right)
= {\mathbf{W}^{(2)}}^\top \frac{\partial J}{\partial \mathbf{o}}.
$$

Vì hàm kích hoạt $\phi$ áp dụng theo từng phần tử, nên việc tính toán gradient $\partial J/\partial \mathbf{z} \in \mathbb{R}^h$ của biến trung gian $\mathbf{z}$ yêu cầu chúng ta sử dụng toán tử nhân theo từng phần tử, được ký hiệu là $\odot$:

$$
\frac{\partial J}{\partial \mathbf{z}}
= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{h}}, \frac{\partial \mathbf{h}}{\partial \mathbf{z}}\right)
= \frac{\partial J}{\partial \mathbf{h}} \odot \phi'\left(\mathbf{z}\right).
$$

Cuối cùng, chúng ta có thể thu được gradient $\partial J/\partial \mathbf{W}^{(1)} \in \mathbb{R}^{h \times d}$ của các tham số mô hình gần lớp đầu vào nhất.
Theo quy tắc chuỗi, chúng ta có được:

$$
\frac{\partial J}{\partial \mathbf{W}^{(1)}}
= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{z}}, \frac{\partial \mathbf{z}}{\partial \mathbf{W}^{(1)}}\right) + \textrm{prod}\left(\frac{\partial J}{\partial s}, \frac{\partial s}{\partial \mathbf{W}^{(1)}}\right)
= \frac{\partial J}{\partial \mathbf{z}} \mathbf{x}^\top + \lambda \mathbf{W}^{(1)}.
$$

## Huấn luyện mạng thần kinh

Khi huấn luyện mạng thần kinh, lan truyền xuôi và lan truyền ngược phụ thuộc lẫn nhau.
Cụ thể, đối với lan truyền xuôi, chúng ta duyệt qua đồ thị tính toán theo hướng phụ thuộc và tính toán tất cả các biến trên đường đi của nó.
Chúng sau đó được sử dụng cho lan truyền ngược trong đó thứ tự tính toán trên đồ thị được đảo ngược.

Hãy lấy mạng đơn giản nói trên làm ví dụ minh họa.
Một mặt, việc tính toán số hạng chuẩn hóa :eqref:`eq_forward-s` trong quá trình lan truyền xuôi phụ thuộc vào các giá trị hiện tại của các tham số mô hình $\mathbf{W}^{(1)}$ và $\mathbf{W}^{(2)}$.
Chúng được đưa ra bởi thuật toán tối ưu hóa theo lan truyền ngược trong lần lặp gần đây nhất.
Mặt khác, việc tính toán gradient cho tham số :eqref:`eq_backprop-J-h` trong quá trình lan truyền ngược phụ thuộc vào giá trị hiện tại của đầu ra lớp ẩn $\mathbf{h}$, được đưa ra bởi lan truyền xuôi.

Do đó, khi huấn luyện mạng thần kinh, một khi các tham số mô hình được khởi tạo, chúng ta xen kẽ lan truyền xuôi với lan truyền ngược, cập nhật các tham số mô hình bằng cách sử dụng các gradient được cung cấp bởi lan truyền ngược.
Lưu ý rằng lan truyền ngược tái sử dụng các giá trị trung gian được lưu trữ từ lan truyền xuôi để tránh các phép tính lặp lại.
Một trong những hậu quả là chúng ta cần giữ lại các giá trị trung gian cho đến khi lan truyền ngược hoàn tất.
Đây cũng là một trong những lý do tại sao việc huấn luyện đòi hỏi nhiều bộ nhớ hơn đáng kể so với việc dự đoán thông thường.
Hơn nữa, kích thước của các giá trị trung gian như vậy tỷ lệ thuận với số lớp mạng và kích thước batch.
Do đó, huấn luyện các mạng sâu hơn bằng cách sử dụng kích thước batch lớn hơn sẽ dễ dẫn đến lỗi *hết bộ nhớ* (out-of-memory).

## Tóm tắt

Lan truyền xuôi tính toán tuần tự và lưu trữ các biến trung gian trong đồ thị tính toán được xác định bởi mạng thần kinh. Nó tiến hành từ lớp đầu vào đến lớp đầu ra.
Lan truyền ngược tính toán tuần tự và lưu trữ các gradient của các biến trung gian và các tham số trong mạng thần kinh theo thứ tự đảo ngược.
Khi huấn luyện các mô hình học sâu, lan truyền xuôi và lan truyền ngược phụ thuộc lẫn nhau, và việc huấn luyện đòi hỏi nhiều bộ nhớ hơn đáng kể so với dự đoán.

## Bài tập

1. Giả sử rằng các đầu vào $\mathbf{X}$ cho một hàm vô hướng $f$ là các ma trận kích thước $n \times m$. Số chiều của gradient của $f$ đối với $\mathbf{X}$ là bao nhiêu?
1. Thêm một số hạng định thiên (bias) vào lớp ẩn của mô hình được mô tả trong phần này (bạn không cần bao gồm bias trong số hạng chuẩn hóa).
    1. Vẽ đồ thị tính toán tương ứng.
    1. Rút ra các phương trình lan truyền xuôi và lan truyền ngược.
1. Tính toán dung lượng bộ nhớ cho việc huấn luyện và dự đoán trong mô hình được mô tả trong phần này.
1. Giả sử rằng bạn muốn tính toán các đạo hàm cấp hai. Điều gì xảy ra với đồ thị tính toán? Bạn kỳ vọng việc tính toán sẽ mất bao lâu?
1. Giả sử rằng đồ thị tính toán quá lớn đối với GPU của bạn.
    1. Bạn có thể phân vùng nó trên nhiều GPU không?
    1. Những ưu điểm và nhược điểm so với việc huấn luyện trên một mini-batch nhỏ hơn là gì?

[Thảo luận](https://discuss.d2l.ai/t/102)